# Performance Tuning

This notebook covers key performance optimization techniques in Spark.

## Learning Objectives

- Understand partitioning
- Use broadcast joins
- Leverage Adaptive Query Execution
- Identify performance bottlenecks

In [ ]:
import sys
sys.path.insert(0, "/opt/spark")

from utils.connect_session import get_connect_url
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, broadcast, spark_partition_id
# Spark Connect: this notebook is a thin client.
# All computation runs on the Spark cluster (see Spark UI).
spark = (
    SparkSession.builder
    .appName("Performance-Tuning")
    .remote(get_connect_url())   # e.g. sc://spark-master:15002
    .getOrCreate()
)


## 1. Partitioning

Partitioning determines how data is distributed across executors.

In [ ]:
# Check current partition count
# (Spark Connect has no RDD API — count distinct spark_partition_id values instead)
df = spark.range(1, 1000000)
num_partitions = df.select(spark_partition_id().alias("pid")).distinct().count()
print(f"Default partitions: {num_partitions}")


In [ ]:
# repartition: Increases or decreases partitions (shuffles data)
df_repartitioned = df.repartition(10)
num_partitions = df_repartitioned.select(spark_partition_id()).distinct().count()
print(f"After repartition(10): {num_partitions} partitions")


In [ ]:
# coalesce: Decreases partitions (no shuffle)
df_coalesced = df_repartitioned.coalesce(2)
num_partitions = df_coalesced.select(spark_partition_id()).distinct().count()
print(f"After coalesce(2): {num_partitions} partitions")


In [ ]:
# Partition by column (useful for joins/aggregations)
try:
    orders = spark.read.parquet("/opt/spark/data/orders/small")

    # Repartition by user_id for user-based operations
    orders_by_user = orders.repartition(10, "user_id")
    num_partitions = orders_by_user.select(spark_partition_id()).distinct().count()
    print(f"Orders repartitioned by user_id: {num_partitions} partitions")

except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")


## 2. Broadcast Joins

Broadcast joins send a small table to all executors, avoiding shuffle.

In [ ]:
# Create a small lookup table
countries = [
    ("USA", "United States", "North America"),
    ("UK", "United Kingdom", "Europe"),
    ("Germany", "Germany", "Europe"),
    ("France", "France", "Europe"),
]

countries_df = spark.createDataFrame(countries, ["code", "name", "region"])
print(f"Countries table size: {countries_df.count()} rows")

In [ ]:
# Broadcast join hint
try:
    users = spark.read.parquet("/opt/spark/data/users/small")
    
    # Without broadcast (shuffle join)
    print("Regular join (with shuffle):")
    regular_join = users.join(countries_df, users.country == countries_df.code)
    regular_join.explain()
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

In [ ]:
# With broadcast (no shuffle)
try:
    print("\nBroadcast join (no shuffle):")
    broadcast_join = users.join(broadcast(countries_df), users.country == countries_df.code)
    broadcast_join.explain()
    
    # Show result
    broadcast_join.select("username", "country", "name").show(5)
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 3. Adaptive Query Execution (AQE)

AQE dynamically optimizes queries at runtime.

In [ ]:
# Check AQE status
print(f"AQE enabled: {spark.conf.get('spark.sql.adaptive.enabled')}")
print(f"Coalesce partitions: {spark.conf.get('spark.sql.adaptive.coalescePartitions.enabled')}")
print(f"Advisory partition size: {spark.conf.get('spark.sql.adaptive.advisoryPartitionSizeInBytes')}")

In [ ]:
# AQE automatically:
# 1. Coalesces small shuffle partitions
# 2. Converts sort-merge joins to broadcast joins when appropriate
# 3. Handles skew joins

try:
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    products = spark.read.parquet("/opt/spark/data/products/small")
    
    # AQE will optimize this join
    result = orders.join(products, "product_id") \
        .groupBy("category") \
        .sum("total_amount")
    
    # Show the plan
    result.explain()
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 4. Reading the Spark UI

The Spark UI is your best tool for performance analysis.

### Key tabs:
1. **Jobs**: Overall job progress
2. **Stages**: Detailed stage information
3. **Storage**: Cached data
4. **Environment**: Configuration
5. **Executors**: Resource usage
6. **SQL**: Query plans

### What to look for:
- Skewed tasks (some tasks much longer than others)
- Shuffle read/write sizes
- Spill to disk
- Number of tasks per stage

In [ ]:
# Run a query and check the UI
try:
    users = spark.read.parquet("/opt/spark/data/users/small")
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    
    # Complex query
    result = users.join(orders, "user_id") \
        .groupBy("country") \
        .agg({"total_amount": "sum"}) \
        .orderBy(col("sum(total_amount)").desc())
    
    result.show()
    
    print("\nCheck the Spark UI at http://localhost:4040")
    print("Look at the SQL tab for the query plan")
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

## 5. Common Performance Issues

### Data Skew
Some partitions have much more data than others.

In [ ]:
# Detecting skew: Check partition sizes
try:
    orders = spark.read.parquet("/opt/spark/data/orders/small")
    
    # Group by status and count
    status_counts = orders.groupBy("status").count().orderBy(col("count").desc())
    status_counts.show()
    
    # If one status dominates, joins on status will be skewed
    
except Exception as e:
    print(f"Data not found. Run 'task generate' first. Error: {e}")

### Solutions for Skew:
1. **Salting**: Add random prefix to keys
2. **AQE**: Enable skew join handling
3. **Broadcast**: Use broadcast join if one side is small

## 6. Best Practices Summary

1. **Use appropriate file formats**: Parquet > CSV/JSON
2. **Partition wisely**: Match your query patterns
3. **Broadcast small tables**: Avoid shuffle when possible
4. **Enable AQE**: Let Spark optimize dynamically
5. **Cache strategically**: Only when reused multiple times
6. **Monitor the UI**: Your best debugging tool

## 7. Exercises

In [ ]:
# Exercise 1: Compare join performance with and without broadcast
# Your code here:


In [ ]:
# Exercise 2: Repartition orders by user_id and measure impact on a join
# Your code here:


In [ ]:
# Exercise 3: Use explain() to compare query plans
# Your code here:


In [ ]:
# Clean up
spark.stop()